<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/M08-deep-learning/AT%26T_logo_2016.svg" alt="AT&T LOGO" width="30%" />

# Orange SPAM detector

In [2]:
import io
import re
import string
import tensorflow as tf
import tqdm
import os
import shutil
import pandas as pd

from tensorflow.keras import Model
from tensorflow.keras.layers import Dot, Embedding, Flatten

import warnings
warnings.filterwarnings('ignore')



2025-08-11 16:36:34.023561: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
import chardet
import requests

url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Deep+Learning/project/spam.csv"
response = requests.get(url)

# Vérifier si la requête a réussi
if response.status_code == 200:
    rawdata = response.content  # Obtenir les données en bytes
    result = chardet.detect(rawdata[:100000])  # Détecter l'encodage
    print(result)
else:
    print("Échec du téléchargement du fichier :", response.status_code)


{'encoding': 'Windows-1252', 'confidence': 0.7272080023536335, 'language': ''}


In [8]:
df = pd.read_csv("https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Deep+Learning/project/spam.csv",encoding='Windows-1252')

In [5]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [6]:
from skimpy import skim
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 5572   │ │ string      │ 5     │                                                          │
│ │ Number of columns │ 5      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓  │
│ ┃          ┃      ┃          ┃          ┃          ┃           ┃          ┃ chars per ┃ words    ┃ total     ┃  │
│ ┃ column   ┃ NA   ┃ NA %     ┃ shortest ┃ longest  ┃ min       ┃ max      ┃ row       ┃ per row  ┃ words     ┃  │
│ ┡━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩  │
│ │ v1       │    0 │        0 │ ham      │ spam     │ ham       │ spam     │      3.13 │        1 │      5572 │  │
│ │ v2       │    0 │        0 │ Ok       │ For me   │  &lt;#&gt │ ‰Û_ we r │      80.1 │       16 │     86961 │  │
│ │          │      │          │          │ the love │ ;  in     │ stayin   │           │          │           │  │
│ │          │      │          │          │ should   │ mca. But  │ here an  │           │          │           │  │
│ │          │      │          │          │ start    │ not       │ extra    │           │          │           │  │
│ │          │      │          │          │ with     │ conform.  │ week,    │           │          │           │  │
│ │          │      │          │          │ attracti │           │ back     │           │          │           │  │
│ │          │      │          │          │ on.i     │           │ next     │           │          │           │  │
│ │          │      │          │          │ should   │           │ wed. How │           │          │           │  │
│ │          │      │          │          │ feel     │           │ did we   │           │          │           │  │
│ │          │      │          │          │ that I   │           │ do in    │           │          │           │  │
│ │          │      │          │          │ need her │           │ the      │           │          │           │  │
│ │          │      │          │          │ every    │           │ rugby    │           │          │           │  │
│ │          │      │          │          │ time     │           │ this     │           │          │           │  │
│ │          │      │          │          │ around   │           │ weekend? │           │          │           │  │
│ │          │      │          │          │ me.she   │           │ Hi to    │           │          │           │  │
│ │          │      │          │          │ should   │           │ and and  │           │          │           │  │
│ │          │      │          │          │ be the   │           │ , c u    │           │          │           │  │
│ │          │      │          │          │ first    │           │ soon     │           │          │           │  │
│ │          │      │          │          │ thing    │           │ \"ham"   │           │          │           │  │
│ │          │      │          │          │ which    │           │          │           │          │           │  │
│ │          │      │          │          │ comes in │  

In [106]:
ham_spam = df['v1'].value_counts()

In [107]:
ham_spam = ham_spam.to_frame()

In [108]:
ham_spam = ham_spam.reset_index()

In [109]:
ham_spam

,v1,count
0,ham,4825
1,spam,747


In [110]:

ham_spam['Target'] = ham_spam['v1'].replace(to_replace=["spam"], value="Spam 🍔")

In [111]:
ham_spam['Target'] = ham_spam['v1'].replace(to_replace=["ham"], value="Ham 🍣")

In [112]:
ham_spam

,v1,count,Target
0,ham,4825,Ham 🍣
1,spam,747,spam


In [113]:
ham_spam['Target'] = ham_spam['Target'].replace(to_replace=["spam"], value="Spam 🍔")

In [114]:
ham_spam

,v1,count,Target
0,ham,4825,Ham 🍣
1,spam,747,Spam 🍔


In [115]:
import plotly.express as px
import plotly.graph_objects as go
explode = [0.1, 0.1]
custom_colors = ['#34ebb4', '#de0000']
fig = go.Figure(go.Pie(
    labels = ham_spam['Target'],
    values = ham_spam['count'],
    pull = explode,
    marker=dict(colors=custom_colors),
    textinfo='percent+label'
))

fig.update_layout(title='Répartition entre <b style="color:#34ebb4">Ham 🍣</b> et <b style="color:#de0000">Spam 🍔</b>')
fig.update_layout(title_font_size=26)
fig.update_layout(width=800)
fig.show()

### N.B :  Les classes sont déséquilibrés entre Ham et Spam

In [116]:
import plotly.graph_objects as go

explode = [0.1, 0.1]
custom_colors = ['#34ebb4', '#de0000']
fig = go.Figure(go.Pie(
    labels=ham_spam['Target'],
    values=ham_spam['count'],
    pull=explode,
    marker=dict(colors=custom_colors),
    texttemplate="<b>%{percent}<br></b> %{label}"
))

fig.update_layout(title='Répartition entre <b style="color:#34ebb4">Ham 🍣</b> et <b style="color:#de0000">Spam 🍔</b>')
fig.update_layout(title_font_size=26)
fig.update_layout(width=800)
fig.show()

In [117]:
df['v1'].shape[0]

5572

In [118]:
c = 747/4825
c

0.15481865284974095

In [119]:
# on regarde le % de données manquantes
unnamed_2 = df['Unnamed: 2'].isna().sum()/df.shape[0] * 100
print(f"Pourcentage de données manquantes pour Unnamed: 2: {round(unnamed_2, 2)} ％ ")
unnamed_3 = df['Unnamed: 3'].isna().sum()/df.shape[0] * 100
print(f"Pourcentage de données manquantes pour Unnamed: 3: {round(unnamed_3, 2)} ％")
unnamed_4 = df['Unnamed: 4'].isna().sum()/df.shape[0] * 100
print(f"Pourcentage de données manquantes pour Unnamed: 4: {round(unnamed_4, 2)} ％")


Pourcentage de données manquantes pour Unnamed: 2: 99.1 ％ 
Pourcentage de données manquantes pour Unnamed: 3: 99.78 ％
Pourcentage de données manquantes pour Unnamed: 4: 99.89 ％


## Supression des colonnes avec données manquantes

In [120]:
df = df[['v1', 'v2']]
df = df.rename(columns={'v1' : 'Target', 'v2' : 'SMS'})


In [121]:
df['Target'] = df['Target'].replace(to_replace=["ham"], value=0)
df['Target'] = df['Target'].replace(to_replace=["spam"], value=1)